In [7]:
from langgraph.graph import StateGraph
from typing import Dict, Any
from custom_llm import CustomLLM

class VacancyAnalyzerAgent:
    def __init__(self, llm: CustomLLM):
        self.llm = llm
        
    def analyze_vacancy(self, state: Dict[str, Any]) -> Dict[str, Any]:
        vacancy_text = state["vacancy_description"]
        
        system_prompt = """You are an experienced HR analyst. Your task is to thoroughly analyze job descriptions 
        and identify key requirements, skills, and company specifics. Pay attention to details."""
        
        prompt = f"""
        Analyze the job description and identify ALL key elements:

        JOB DESCRIPTION:
        {vacancy_text}

        RETURN RESPONSE IN THE FORMAT:
        - Position: [job title]
        - Key responsibilities: [list of responsibilities]
        - Required skills: [list of skills]
        - Preferred skills: [list of skills] 
        - Required experience: [experience level]
        - Language level: [if specified]
        - Company/project specifics: [what stands out]
        - Technology stack: [technologies and tools]
        """
        
        analysis = self.llm.invoke(prompt, system_prompt)
        return {"vacancy_analysis": analysis}
    
    def analyze_resume(self, state: Dict[str, Any]) -> Dict[str, Any]:
        resume_text = state["resume"]
        
        system_prompt = """You are a professional HR expert. Analyze candidate resumes, 
        highlighting their key competencies, experience, and achievements. Be objective and attentive."""
        
        prompt = f"""
        Analyze the candidate's resume:

        RESUME:
        {resume_text}

        RETURN RESPONSE IN THE FORMAT:
        - Key skills: [list of main skills]
        - Work experience: [overall experience and key positions]
        - Achievements: [specific results and achievements]
        - Education: [education and certificates]
        - Language level: [if specified]
        - Strengths: [what distinguishes the candidate]
        - Projects: [key projects if available]
        """
        
        analysis = self.llm.invoke(prompt, system_prompt)
        return {"resume_analysis": analysis}
        
    def create_match_report(self, state: Dict[str, Any]) -> Dict[str, Any]:
        vacancy_analysis = state.get("vacancy_analysis", "")
        resume_analysis = state.get("resume_analysis", "")
        
        system_prompt = """You are a recruitment expert. Compare job requirements 
        and candidate competencies to create an objective and useful report for composing 
        a cover letter. Be specific and provide practical recommendations."""
        
        prompt = f"""
        Based on the analysis of the vacancy and resume, create a DETAILED REPORT for the cover letter generator.

        VACANCY ANALYSIS:
        {vacancy_analysis}

        RESUME ANALYSIS:
        {resume_analysis}

        CREATE THE REPORT IN THE FOLLOWING FORMAT:

        ## STRONG SIDES (perfect match):
        - [specific skill/experience from resume] → [corresponding job requirement]
        - [another matching point]

        ## GROWTH AREAS (partial match):
        - [skill that exists but needs development] → [job requirement]
        - [experience that can be presented differently]

        ## GAPS (missing requirements):
        - [what is missing in the resume] → [job requirement]

        ## RECOMMENDATIONS FOR COVER LETTER:
        ### What to emphasize:
        - [specific points to highlight]
        
        ### How to compensate for gaps:
        - [strategies to explain missing skills]
        
        ### Keywords to use:
        - [words and phrases from the vacancy]
        
        ### Letter tone:
        - [recommended tone: confident/enthusiastic/professional]
        """
        
        report = self.llm.invoke(prompt, system_prompt)
        return {"analysis_report": report}

In [8]:
from typing import TypedDict

class State(TypedDict):
    vacancy_description: str
    resume: str
    vacancy_analysis: str
    resume_analysis: str
    analysis_report: str

def create_analysis_workflow():
    custom_llm = CustomLLM()
    
    analyzer_agent = VacancyAnalyzerAgent(custom_llm)
    
    workflow = StateGraph(State)

    workflow.add_node("analyze_vacancy", analyzer_agent.analyze_vacancy)
    workflow.add_node("analyze_resume", analyzer_agent.analyze_resume)
    workflow.add_node("create_report", analyzer_agent.create_match_report)
    
    workflow.set_entry_point("analyze_vacancy")
    workflow.add_edge("analyze_vacancy", "analyze_resume")
    workflow.add_edge("analyze_resume", "create_report")
    workflow.set_finish_point("create_report")
    
    return workflow.compile()

analysis_workflow = create_analysis_workflow()

In [9]:
def analyze_vacancy_match(vacancy_text: str, resume_text: str) -> Dict[str, Any]:
    initial_state = {
        "vacancy_description": vacancy_text,
        "resume": resume_text
    }
    
    result = analysis_workflow.invoke(initial_state)
    
    return result

In [10]:
def load_vacancy_from_txt(file_path: str) -> str:
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            return file.read().strip()
    except FileNotFoundError:
        print(f"File with vacancy {file_path} doesn't found")
        return ""
    except Exception as e:
        print(f"Error reading file with vacancy: {e}")
        return ""

def load_resume_from_md(file_path: str) -> str:
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            return file.read().strip()
    except FileNotFoundError:
        print(f"File with resume {file_path} doesn't found")
        return ""
    except Exception as e:
        print(f"Error reading the file with resume: {e}")
        return ""

In [ ]:
vacancy_file_path = "results/vacancy.txt"  
resume_file_path = "resume.md"     

print("Loading vacancy and resume")
vacancy_text = load_vacancy_from_txt(vacancy_file_path)
resume_text = load_resume_from_md(resume_file_path)

print(f"Vacancy uploaded ({len(vacancy_text)} chars)")
print(f"Resume uploaded ({len(resume_text)} chars)")

print("Start the ai-agent...")
report = analyze_vacancy_match(vacancy_text, resume_text)

Loading vacancy and resume
Vacancy uploaded (3099 chars)
Resume uploaded (10310 chars)
Start the ai-agent...
Error when calling LLM: 429 Client Error: Too Many Requests for url: https://api.intelligence.io.solutions/api/v1/chat/completions
Error when calling LLM: 429 Client Error: Too Many Requests for url: https://api.intelligence.io.solutions/api/v1/chat/completions
Error when calling LLM: 429 Client Error: Too Many Requests for url: https://api.intelligence.io.solutions/api/v1/chat/completions


In [12]:
output_file = "analysis_report.txt"
with open(output_file, "w", encoding="utf-8") as f:
    f.write(report["analysis_report"])

print(f"Analysis report saved in: {output_file}")

Analysis report saved in: analysis_report.txt
